# Attention Head Output Analysis

Analyze what each head writes: writing direction, unembedding alignment,
diversity, position specialization, and combined effect.

## Why This Matters

Attention head output reveals how heads route information between positions. Understanding attention mechanics is central to reverse-engineering transformer circuits, as attention patterns determine what information each component can access and transform.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.attention_head_output_analysis import (
    head_writing_direction, head_unembed_alignment,
    head_output_diversity, head_position_specialization,
    head_combined_effect,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Writing Direction

What direction does each head write, and how consistent is it across positions?

In [ ]:
for layer in range(4):
    result = head_writing_direction(model, tokens, layer=layer)
    print(f"Layer {layer} ({result['n_consistent']} consistent):")
    for h in result['per_head']:
        cons = '✓' if h['is_consistent'] else '✗'
        print(f"  H{h['head']}: norm={h['mean_output_norm']:.4f}, "
              f"consistency={h['direction_consistency']:.4f} [{cons}]")
    print()

## Unembedding Alignment

Which heads write in directions aligned with specific token embeddings?

In [ ]:
result = head_unembed_alignment(model, tokens, layer=3, position=-1)
print(f"Layer {result['layer']}, position {result['position']}:\n")
for h in result['per_head']:
    spec = ' [TOKEN-SPECIFIC]' if h['is_token_specific'] else ''
    print(f"  H{h['head']}: max_align={h['max_alignment']:.4f} "
          f"(token {h['max_aligned_token']}), "
          f"mean_abs={h['mean_abs_alignment']:.4f}{spec}")

## Output Diversity

How different are the outputs of heads within a layer?

In [ ]:
for layer in range(4):
    result = head_output_diversity(model, tokens, layer=layer)
    div = 'DIVERSE' if result['is_diverse'] else 'similar'
    print(f"Layer {layer}: mean_abs_sim={result['mean_abs_similarity']:.4f} [{div}]")
    for p in result['pairs']:
        print(f"    H{p['head_a']}↔H{p['head_b']}: {p['similarity']:+.4f}")
    print()

## Position Specialization

Do heads focus their output on specific positions?

In [ ]:
for layer in range(4):
    result = head_position_specialization(model, tokens, layer=layer)
    print(f"Layer {layer} ({result['n_specialized']} specialized):")
    for h in result['per_head']:
        spec = ' [SPECIALIZED]' if h['is_position_specialized'] else ''
        print(f"  H{h['head']}: cv={h['coefficient_of_variation']:.4f}, "
              f"max_pos={h['max_position']}{spec}")
    print()

## Combined Effect

How do all heads combine? Constructive or destructive interference?

In [ ]:
for layer in range(4):
    result = head_combined_effect(model, tokens, layer=layer)
    print(f"Layer {layer}: combined={result['combined_norm']:.4f}, "
          f"eff={result['efficiency']:.2%}, "
          f"construct={result['total_constructive']:.4f}, "
          f"destruct={result['total_destructive']:.4f}")
    for h in result['per_head']:
        sign = '+' if h['is_constructive'] else '-'
        print(f"  H{h['head']}: proj={h['projection_onto_combined']:+.4f} [{sign}]")
    print()